# NB_Purview_Copilot_BronzeToSilver
**Source:** `AI_Medallion.Bronze.purview_copilot_usage`  
**Targets (reads):** `okta_user_for_ai`, `hc_silver_historical`  
**Target:** `AI_Medallion.Silver.purview_copilot_usage` (incremental merge on `purview_copilot_usage_record_id`)  
**Run after:** `NB_OktaUserforAI_Bronze_To_Silver`, `NB_HCHistorical_Bronze_To_Silver`  
**Local:** read uses `~/.fabric/medallion_cache` (full bronze). Filter **`CreationDate`** in the transform cell — do not push month filters to deltalake scan (blob GETs often fail on Windows).

In [ ]:
# Workspace folder: C:\spark-dev\CoE_transformation_framework (see README)
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "25000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# OneLake blob GETs: serial reads avoid "error sending request" on large bronze (many ~150MB files).
os.environ.setdefault("MEDALLION_OBJECT_STORE_CONCURRENCY", "1")
# Do not set MEDALLION_CACHE_REFRESH=1 unless you must re-pull all of bronze from OneLake.

# CreationDate window (applied in transform, not at OneLake scan)
PURVIEW_BRONZE_CREATION_MONTH = None
# PURVIEW_BRONZE_CREATION_MONTH = "2026-09"
PURVIEW_BRONZE_FILENAMES = []


In [ ]:
%run ./Common/bootstrap


In [ ]:
bronze = read_table_broken_lineage(TABLE_PURVIEW_COPILOT_BRONZE, "purview_copilot_bronze")


In [ ]:
okta = read_table_broken_lineage(TABLE_OKTA_SILVER, "okta_silver")


In [ ]:
hc = read_table_broken_lineage(TABLE_HC_SILVER, "hc_silver")


In [ ]:
apply_local_merge_spark_conf(spark)

_creation_col = (
    PURVIEW_BRONZE_FILTER_DATE_COLUMN
    if PURVIEW_BRONZE_FILTER_DATE_COLUMN in bronze.columns
    else "creation_date"
)
_file_col = "FileName" if "FileName" in bronze.columns else "file_name"

if PURVIEW_BRONZE_CREATION_MONTH:
    _month_start = F.to_date(F.lit(f"{PURVIEW_BRONZE_CREATION_MONTH}-01"))
    _month_end = F.add_months(_month_start, 1)
    _creation_date = F.to_timestamp(F.trim(F.col(_creation_col))).cast("date")
    bronze = bronze.filter(
        (_creation_date >= _month_start) & (_creation_date < _month_end)
    )
    min_date = f"{PURVIEW_BRONZE_CREATION_MONTH}-01"
    print(
        f"purview bronze: CreationDate in {PURVIEW_BRONZE_CREATION_MONTH}",
        flush=True,
    )
else:
    min_date = os.environ.get(
        "MEDALLION_PURVIEW_MIN_CREATION_DATE", PURVIEW_COPILOT_MIN_CREATION_DATE
    )

if PURVIEW_BRONZE_FILENAMES:
    bronze = bronze.filter(F.col(_file_col).isin(PURVIEW_BRONZE_FILENAMES))
    print(
        f"purview bronze: FileName filter ({len(PURVIEW_BRONZE_FILENAMES)} file(s))",
        flush=True,
    )

print(f"purview_copilot_usage: creation_date >= {min_date}")

write_df = build_purview_copilot_usage_silver(
    bronze,
    okta,
    hc,
    min_creation_date=min_date,
)

write_df = break_lineage_local(write_df, "purview_copilot_silver")
publish_merge_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)


In [ ]:
merge_incremental(TABLE_PURVIEW_COPILOT_SILVER, PURVIEW_COPILOT_MERGE_KEY)
